# Final Visualization Dataset, Half Corpus

Final dataset workflow for the frontend visualization build.

Dataset strategy:

```text
frontend dataset =
  spec-0006 CURATED_GROUND como um/uma/uns/umas VEHICLE
  + bare-como candidates where spaCy/lexicon label = visual_head_whitelist
```

The pipeline keeps Spark responsible for corpus-scale discovery and uses spaCy only on the extracted bare-`como` candidate windows. It writes one final frontend-ready namespace:

```text
data/gold/visualization/ground_vehicle_candidates
outputs/visualization/candidate_counts.csv
outputs/visualization/ground_vehicle_counts.csv
outputs/visualization/vehicle_ground_counts.csv
outputs/visualization/ground_counts.csv
outputs/visualization/vehicle_counts.csv
outputs/visualization/examples.csv
outputs/visualization/review_sample.csv
```


In [1]:
import os
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src/tal_qual").exists():
            return candidate
        if (candidate / "work/src/tal_qual").exists():
            return candidate / "work"
    raise RuntimeError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPO_ROOT


PosixPath('/home/jovyan/work')

In [2]:
import tempfile
import zipfile

import pandas as pd
import spacy
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

SPARK_MASTER = os.environ.get("TAL_QUAL_SPARK_MASTER", "local[4]")
SPARK_PARALLELISM = os.environ.get("TAL_QUAL_SPARK_PARALLELISM", "4")
SPARK_SHUFFLE_PARTITIONS = os.environ.get("TAL_QUAL_SPARK_SHUFFLE_PARTITIONS", "4")
SPARK_DRIVER_MEMORY = os.environ.get("TAL_QUAL_SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .master(SPARK_MASTER)
    .appName("tal-qual-final-visualization-half-corpus")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .config("spark.default.parallelism", SPARK_PARALLELISM)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTITIONS)
    .config("spark.sql.files.maxPartitionBytes", str(128 * 1024 * 1024))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

package_zip = Path(tempfile.gettempdir()) / "tal_qual_src.zip"
with zipfile.ZipFile(package_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted((SRC_DIR / "tal_qual").rglob("*.py")):
        archive.write(file_path, file_path.relative_to(SRC_DIR))
spark.sparkContext.addPyFile(str(package_zip))

SPARK_MASTER, spark.version


('local[4]', '4.1.1')

In [3]:
from tal_qual.bare_como_spacy_filter import classify_bare_como_candidate
from tal_qual.bronze import BRONZE_OUTPUT_PATH, RAW_CORPUS_INPUT, load_or_build_bronze_dataframe
from tal_qual.como_article_ground_vehicle import (
    prefilter_como_article_ground_vehicle_bronze_dataframe,
    prepare_como_article_ground_vehicle_dataframe,
)
from tal_qual.dataset_expansion_experiments import (
    prefilter_dataset_expansion_bronze_dataframe,
    prepare_dataset_expansion_dataframe,
)


## Load Bronze


In [4]:
default_raw_path = "data/raw/brwac-clean-[1-6].txt.gz"
default_bronze_path = "data/bronze/brwac_segments_half"
raw_path = REPO_ROOT / os.environ.get("TAL_QUAL_RAW_CORPUS_INPUT", default_raw_path)
bronze_path = REPO_ROOT / os.environ.get("TAL_QUAL_BRONZE_PATH", default_bronze_path)

bronze_df = load_or_build_bronze_dataframe(spark, raw_path, bronze_path)
print(f"Raw path: {raw_path}")
print(f"Bronze path: {bronze_path}")
bronze_df.printSchema()


Raw path: /home/jovyan/work/data/raw/brwac-clean-[1-6].txt.gz
Bronze path: /home/jovyan/work/data/bronze/brwac_segments_half
root
 |-- source_file: string (nullable = true)
 |-- original_line_id: long (nullable = true)
 |-- segment_id: integer (nullable = true)
 |-- text_original: string (nullable = true)
 |-- text_normalized: string (nullable = true)
 |-- match_text: string (nullable = true)



## Extract Spec-0006 Baseline


In [5]:
baseline_prefiltered_df = prefilter_como_article_ground_vehicle_bronze_dataframe(bronze_df).persist()
print(f"Spec-0006 prefiltered rows: {baseline_prefiltered_df.count():,}")

baseline_df = prepare_como_article_ground_vehicle_dataframe(baseline_prefiltered_df).persist()
print(f"Spec-0006 candidate rows: {baseline_df.count():,}")


Spec-0006 prefiltered rows: 1,819


/usr/local/spark/python/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Spec-0006 candidate rows: 1,080


## Extract Bare-`como` Candidates And Promote Visual Whitelist


In [6]:
BARE_COMO_BASE_SLUG = "bare_como_curated_ground_v2"
BARE_COMO_PROMOTED_PATTERN = "bare_como_visual_whitelist"

bare_prefiltered_df = prefilter_dataset_expansion_bronze_dataframe(bronze_df, BARE_COMO_BASE_SLUG).persist()
print(f"Bare-como prefiltered rows: {bare_prefiltered_df.count():,}")

bare_candidates_df = prepare_dataset_expansion_dataframe(bare_prefiltered_df, BARE_COMO_BASE_SLUG).persist()
print(f"Bare-como candidate rows: {bare_candidates_df.count():,}")

nlp = spacy.load("pt_core_news_sm", disable=["ner"])
bare_pdf = bare_candidates_df.toPandas()
texts = bare_pdf["candidate_full_text"].fillna("").astype(str).tolist()
records = bare_pdf.to_dict("records")

nlp_rows = []
for record, doc in zip(records, nlp.pipe(texts, batch_size=500)):
    result = classify_bare_como_candidate(record, doc)
    row = dict(record)
    row.update(result)
    nlp_rows.append(row)

bare_nlp_pdf = pd.DataFrame(nlp_rows)
print(bare_nlp_pdf["nlp_quality_reason"].value_counts(dropna=False).head(20))

promoted_bare_pdf = bare_nlp_pdf[bare_nlp_pdf["nlp_quality_reason"] == "visual_head_whitelist"].copy()
promoted_bare_pdf["pattern_type"] = BARE_COMO_PROMOTED_PATTERN
print(f"Bare-como promoted visual-whitelist rows: {len(promoted_bare_pdf):,}")

promoted_bare_df = spark.createDataFrame(promoted_bare_pdf).persist() if len(promoted_bare_pdf) else spark.createDataFrame([], bare_candidates_df.schema)


Bare-como prefiltered rows: 7,009


PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/tmp/spark-2d2ae559-171a-49af-a00a-8da06bdc9400/userFiles-64a1b542-380d-4e1d-a257-db97e31de1e2/tal_qual_src.zip/tal_qual/dataset_expansion_experiments.py", line 333, in extract_dataset_expansion_rows
    "ground_lemma": _GROUND_FORMS[ground_text.lower()],
                    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
KeyError: 'andes c'


## Normalize And Merge


In [ ]:
VISUALIZATION_DATASET_PATH = Path("data/gold/visualization/ground_vehicle_candidates")
VISUALIZATION_OUTPUT_ROOT = Path("outputs/visualization")

baseline_norm_df = baseline_df.select(
    "candidate_id",
    "pattern_type",
    "source_file",
    "original_line_id",
    "segment_id",
    "connector_text",
    "matched_text",
    "candidate_full_text",
    "text_before",
    "ground_text",
    "ground_lemma",
    "ground_source",
    F.col("vehicle_text").alias("vehicle_text_raw"),
    F.col("vehicle_text").alias("vehicle_text_clean"),
    F.col("vehicle_head_lemma").alias("vehicle_head_clean"),
    F.lit("").alias("vehicle_tail_text"),
    F.lit("keep").alias("quality_label"),
    F.lit("spec_0006_strict_filter").alias("quality_reason"),
    F.lit("keep").alias("nlp_quality_label"),
    F.lit("spec_0006_baseline").alias("nlp_quality_reason"),
    F.col("vehicle_head_lemma").alias("spacy_vehicle_head_lemma"),
    F.lit("").alias("spacy_vehicle_head_pos"),
    F.lit(False).alias("nlp_needs_review"),
    "needs_review",
    "char_start",
    "char_end",
)

bare_norm_df = promoted_bare_df.select(
    "candidate_id",
    "pattern_type",
    "source_file",
    "original_line_id",
    "segment_id",
    "connector_text",
    "matched_text",
    "candidate_full_text",
    "text_before",
    "ground_text",
    "ground_lemma",
    "ground_source",
    "vehicle_text_raw",
    "vehicle_text_clean",
    "vehicle_head_clean",
    "vehicle_tail_text",
    "quality_label",
    "quality_reason",
    "nlp_quality_label",
    "nlp_quality_reason",
    "spacy_vehicle_head_lemma",
    "spacy_vehicle_head_pos",
    "nlp_needs_review",
    "needs_review",
    "char_start",
    "char_end",
)

visualization_df = baseline_norm_df.unionByName(bare_norm_df).dropDuplicates(["candidate_id"]).persist()
print(f"Final visualization rows: {visualization_df.count():,}")
visualization_df.groupBy("pattern_type").count().orderBy("pattern_type").show(truncate=False)


## Write Frontend Outputs


In [ ]:
(REPO_ROOT / VISUALIZATION_OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
visualization_df.write.mode("overwrite").parquet(str(REPO_ROOT / VISUALIZATION_DATASET_PATH))

(
    visualization_df.groupBy("pattern_type")
    .count()
    .orderBy("pattern_type")
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / VISUALIZATION_OUTPUT_ROOT / "candidate_counts.csv"))
)
(
    visualization_df.groupBy("ground_lemma", "vehicle_head_clean")
    .count()
    .orderBy(F.col("count").desc(), "ground_lemma", "vehicle_head_clean")
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / VISUALIZATION_OUTPUT_ROOT / "ground_vehicle_counts.csv"))
)
(
    visualization_df.groupBy("vehicle_head_clean", "ground_lemma")
    .count()
    .orderBy(F.col("count").desc(), "vehicle_head_clean", "ground_lemma")
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / VISUALIZATION_OUTPUT_ROOT / "vehicle_ground_counts.csv"))
)
(
    visualization_df.groupBy("ground_lemma")
    .count()
    .orderBy(F.col("count").desc(), "ground_lemma")
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / VISUALIZATION_OUTPUT_ROOT / "ground_counts.csv"))
)
(
    visualization_df.groupBy("vehicle_head_clean")
    .count()
    .orderBy(F.col("count").desc(), "vehicle_head_clean")
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / VISUALIZATION_OUTPUT_ROOT / "vehicle_counts.csv"))
)
(
    visualization_df.select(
        "pattern_type", "ground_text", "ground_lemma", "connector_text",
        "vehicle_text_clean", "vehicle_head_clean", "candidate_full_text",
    )
    .orderBy("ground_lemma", "vehicle_head_clean", "pattern_type", "candidate_id")
    .limit(5000)
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / VISUALIZATION_OUTPUT_ROOT / "examples.csv"))
)
(
    visualization_df.select(
        "candidate_id", "pattern_type", "ground_text", "ground_lemma", "connector_text",
        "vehicle_text_raw", "vehicle_text_clean", "vehicle_head_clean",
        "quality_label", "quality_reason", "nlp_quality_label", "nlp_quality_reason",
        "spacy_vehicle_head_lemma", "spacy_vehicle_head_pos", "candidate_full_text",
    )
    .orderBy("pattern_type", "ground_lemma", "vehicle_head_clean", "candidate_id")
    .limit(1000)
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / VISUALIZATION_OUTPUT_ROOT / "review_sample.csv"))
)


## Inspect Final Dataset


In [ ]:
visualization_df.groupBy("pattern_type").count().orderBy("pattern_type").show(truncate=False)

visualization_df.groupBy("ground_lemma", "vehicle_head_clean").count().orderBy(F.col("count").desc(), "ground_lemma", "vehicle_head_clean").show(60, truncate=False)

print(f"Dataset: {VISUALIZATION_DATASET_PATH}")
print(f"Outputs: {VISUALIZATION_OUTPUT_ROOT}")
